# Restvolumen je Projekt

Schritt 1 aus Spec Abschnitt 10: `/v4/projects` und `/v2/entrygroups` abfragen und
das Restvolumen je Projekt berechnen (Abschnitt 5.1) – als Zwischenschritt **vor**
der Monte-Carlo-Logik.

> **Vor dem ersten Lauf lesen:** Die JSON-Struktur und die Query-Parameter der beiden
> Endpunkte sind nicht gegen die Clockodo-Doku verifiziert – `docs.clockodo.com` wird
> als JavaScript-Anwendung ausgeliefert und war nicht auslesbar. Die Feldnamen
> (`budget.amount`, `revenue`) stammen aus der Spec, das umgebende Envelope ist
> geraten. Die Zellen geben deshalb die Roh-Keys aus; diese prüfen und die
> Extraktion unten anpassen. Stellen dazu sind mit `PRÜFEN` markiert.

In [ ]:
# Nur in Google Colab: Projekt installieren. Lokal passiert hier nichts,
# dort liefert `uv sync` die Umgebung.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install --quiet httpx pandas python-dotenv
    # Paket aus dem Repository nachziehen, z. B.:
    # !pip install --quiet git+https://<host>/<pfad>/umsatzprognose-clockodo.git

IN_COLAB

In [ ]:
import httpx
import pandas as pd

from umsatzprognose.config import BASE_URL, load_credentials
from umsatzprognose.restvolumen import restvolumen_je_projekt, summe_prognosewirksam

if IN_COLAB:
    # In Colab die Werte aus der Secrets-Verwaltung in die Umgebung heben,
    # dann `use_dotenv=False`. Keine .env in Colab anlegen.
    import os

    from google.colab import userdata

    for key in (
        "CLOCKODO_API_USER",
        "CLOCKODO_API_KEY",
        "CLOCKODO_APP_NAME",
        "CLOCKODO_APP_EMAIL",
    ):
        os.environ[key] = userdata.get(key)

creds = load_credentials(use_dotenv=not IN_COLAB)
print("Angemeldet als", creds.api_user)

In [ ]:
def get(path: str, **params):
    """Ein GET gegen die Clockodo-API. Wirft bei HTTP-Fehlern."""
    with httpx.Client(base_url=BASE_URL, headers=creds.headers(), timeout=30.0) as client:
        response = client.get(path, params=params or None)
        response.raise_for_status()
        return response.json()


def zeige_struktur(name: str, payload) -> None:
    """Top-Level-Struktur ausgeben, um das Envelope zu identifizieren."""
    if isinstance(payload, dict):
        print(f"{name}: dict mit Keys {sorted(payload)}")
    elif isinstance(payload, list):
        print(f"{name}: Liste mit {len(payload)} Eintraegen")
        if payload:
            print(f"  erster Eintrag, Keys: {sorted(payload[0])}")
    else:
        print(f"{name}: {type(payload)}")

## Auftragsvolumen aus `/v4/projects`

In [ ]:
projects_raw = get("/v4/projects")
zeige_struktur("projects_raw", projects_raw)

In [ ]:
# PRÜFEN: Envelope-Key und ggf. Paginierung anhand der Ausgabe oben anpassen.
PROJECTS_KEY = "projects"

projects = projects_raw[PROJECTS_KEY] if isinstance(projects_raw, dict) else projects_raw


def projekt_id(projekt: dict) -> int:
    # PRÜFEN: Feldname der Projekt-ID in v4.
    for key in ("id", "projects_id"):
        if key in projekt:
            return int(projekt[key])
    raise KeyError(f"Keine Projekt-ID gefunden, vorhandene Keys: {sorted(projekt)}")


# budget.amount laut Spec Abschnitt 4. budget.hard ist hier false, das Budget ist
# also eine weiche Grenze - Ueberschreitungen sind moeglich und erwartet.
budgets = {
    projekt_id(p): (p.get("budget") or {}).get("amount")
    for p in projects
}

print(f"{len(budgets)} Projekte, davon {sum(v is None for v in budgets.values())} ohne Budget")

## Verbrauchtes Volumen aus `/v2/entrygroups`

Gruppierung nach Projekt, Zeitraum weit genug, um die gesamte Projekthistorie zu
erfassen – `revenue_kumuliert` in Abschnitt 5.1 ist der Gesamtverbrauch, nicht der
eines Monats.

In [ ]:
# PRÜFEN: Parametrisierung von /v2/entrygroups (Name des Gruppierungsparameters,
# Format der Zeitgrenzen, Pflichtfelder). Diese Werte sind nicht verifiziert.
ZEITRAUM_VON = "2020-01-01T00:00:00Z"
ZEITRAUM_BIS = "2026-12-31T23:59:59Z"

entrygroups_raw = get(
    "/v2/entrygroups",
    time_since=ZEITRAUM_VON,
    time_until=ZEITRAUM_BIS,
    grouping="projects_id",
)
zeige_struktur("entrygroups_raw", entrygroups_raw)

In [ ]:
# PRÜFEN: Envelope-Key und Feldnamen der Gruppen anhand der Ausgabe oben.
ENTRYGROUPS_KEY = "groups"

groups = (
    entrygroups_raw[ENTRYGROUPS_KEY]
    if isinstance(entrygroups_raw, dict)
    else entrygroups_raw
)

revenue_kumuliert = {
    int(g["group"]): float(g["revenue"] or 0.0)
    for g in groups
}

print(f"Verbrauch fuer {len(revenue_kumuliert)} Projekte geladen")

## Restvolumen (Spec 5.1)

`roh` ist `budget.amount - revenue_kumuliert` und kann negativ sein, weil
`budget.hard` false ist. `prognosewirksam` kappt bei 0 – nur dieser Teil kann noch
abgerufen werden und geht in die Simulation ein.

In [ ]:
restvolumina, ohne_budget = restvolumen_je_projekt(budgets, revenue_kumuliert)

df = pd.DataFrame(
    [
        {
            "projects_id": r.projects_id,
            "budget": r.budget,
            "revenue_kumuliert": r.revenue_kumuliert,
            "restvolumen_roh": r.roh,
            "prognosewirksam": r.prognosewirksam,
            "ueberschritten": r.ueberschritten,
        }
        for r in restvolumina
    ]
).sort_values("prognosewirksam", ascending=False)

print(f"Prognosewirksames Restvolumen gesamt: {summe_prognosewirksam(restvolumina):,.2f} EUR")
print(f"Projekte mit Budgetueberschreitung:   {int(df['ueberschritten'].sum())}")
print(f"Projekte ohne Budget (ausgeschlossen): {len(ohne_budget)} -> {ohne_budget}")

df.head(30)

## Offen

- Projekte ohne Budget haben kein bezifferbares Auftragsvolumen und fallen hier
  heraus. Ob sie in der Prognose anders behandelt werden sollen, deckt die Spec nicht ab.
- Die Normalisierung von Pauschalleistungen über den effektiven Stundensatz (5.1)
  fehlt noch – die Definition steht in Spec v0.3, die nicht im Repository liegt.
- Nächster Schritt laut Abschnitt 10: Abrufquoten-Verteilungen je Referenzklasse aus
  der `entrygroups`-Historie schätzen.